In [4]:
import pandas as pd

# Load with low_memory=False — IBTrACS has mixed dtype columns
df = pd.read_csv(
    "/data/raw/noaa_ibtracs_raw.csv",
    skiprows=[1],        # Row 1 is a units row, not data
    low_memory=False,
    na_values=[" ", ""]
)

# Preview structure
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(127569, 174)
['SID', 'SEASON', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON', 'WMO_WIND', 'WMO_PRES', 'WMO_AGENCY', 'TRACK_TYPE', 'DIST2LAND', 'LANDFALL', 'IFLAG', 'USA_AGENCY', 'USA_ATCF_ID', 'USA_LAT', 'USA_LON', 'USA_RECORD', 'USA_STATUS', 'USA_WIND', 'USA_PRES', 'USA_SSHS', 'USA_R34_NE', 'USA_R34_SE', 'USA_R34_SW', 'USA_R34_NW', 'USA_R50_NE', 'USA_R50_SE', 'USA_R50_SW', 'USA_R50_NW', 'USA_R64_NE', 'USA_R64_SE', 'USA_R64_SW', 'USA_R64_NW', 'USA_POCI', 'USA_ROCI', 'USA_RMW', 'USA_EYE', 'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_GRADE', 'TOKYO_WIND', 'TOKYO_PRES', 'TOKYO_R50_DIR', 'TOKYO_R50_LONG', 'TOKYO_R50_SHORT', 'TOKYO_R30_DIR', 'TOKYO_R30_LONG', 'TOKYO_R30_SHORT', 'TOKYO_LAND', 'CMA_LAT', 'CMA_LON', 'CMA_CAT', 'CMA_WIND', 'CMA_PRES', 'HKO_LAT', 'HKO_LON', 'HKO_CAT', 'HKO_WIND', 'HKO_PRES', 'KMA_LAT', 'KMA_LON', 'KMA_CAT', 'KMA_WIND', 'KMA_PRES', 'KMA_R50_DIR', 'KMA_R50_LONG', 'KMA_R50_SHORT', 'KMA_R30_DIR', 'KMA_R30_LONG', 'KMA_R30_SHORT', 'NEWDELHI_LAT', 'NEW

In [5]:
# Filter to 2019-2024 and preview key columns only
df_filtered = df[df["SEASON"].astype(str).str[:4].astype(int).between(2019, 2024)].copy()

print("Rows after year filter:", len(df_filtered))
print("Unique storms:", df_filtered["SID"].nunique())
print("\nSEASON value counts:")
print(df_filtered["SEASON"].value_counts().sort_index())

print("\nUSA_SSHS value counts (storm category):")
print(df_filtered["USA_SSHS"].value_counts().sort_index())

print("\nSample LANDFALL values:")
print(df_filtered["LANDFALL"].value_counts().head(10))

print("\nNature values:")
print(df_filtered["NATURE"].value_counts())

Rows after year filter: 7182
Unique storms: 130

SEASON value counts:
SEASON
2019     948
2020    1699
2021    1163
2022     913
2023    1523
2024     936
Name: count, dtype: int64

USA_SSHS value counts (storm category):
USA_SSHS
-4    1022
-3    1092
-2     179
-1     884
 0    2522
 1     688
 2     330
 3     222
 4     206
 5      37
Name: count, dtype: int64

Sample LANDFALL values:
LANDFALL
0.0      838
22.0      36
11.0      32
44.0      20
10.0      19
30.0      19
55.0      18
33.0      17
67.0      16
111.0     16
Name: count, dtype: int64

Nature values:
NATURE
TS    4889
DS    1092
ET    1022
SS     179
Name: count, dtype: int64


In [6]:
df_filtered = df[df["SEASON"].astype(str).str[:4].astype(int).between(2019, 2024)].copy()

# Keep only relevant columns and check coordinate ranges
cols = ["SID", "SEASON", "NAME", "ISO_TIME", "NATURE", 
        "LAT", "LON", "USA_WIND", "USA_SSHS", "DIST2LAND", "LANDFALL"]

df_sub = df_filtered[cols].copy()

print("LAT range:", df_sub["LAT"].min(), "to", df_sub["LAT"].max())
print("LON range:", df_sub["LON"].min(), "to", df_sub["LON"].max())

print("\nUSA_WIND sample stats:")
print(df_sub["USA_WIND"].describe())

print("\nRows where LANDFALL == 0:")
print(len(df_sub[df_sub["LANDFALL"] == 0]))

print("\nSample of near-landfall rows:")
print(df_sub[df_sub["LANDFALL"] == 0][["SID", "NAME", "ISO_TIME", "LAT", "LON", "USA_WIND", "USA_SSHS"]].head(10).to_string())

LAT range: 7.0 to 64.0
LON range: -136.9 to 8.0

USA_WIND sample stats:
count    7182.000000
mean       49.644389
std        24.505139
min        15.000000
25%        31.500000
50%        43.000000
75%        60.000000
max       160.000000
Name: USA_WIND, dtype: float64

Rows where LANDFALL == 0:
838

Sample of near-landfall rows:
                  SID   NAME             ISO_TIME   LAT   LON  USA_WIND  USA_SSHS
119693  2019192N29274  BARRY  2019-07-13 12:00:00  29.3 -91.9      65.0         1
119694  2019192N29274  BARRY  2019-07-13 15:00:00  29.6 -92.2      65.0         1
119695  2019192N29274  BARRY  2019-07-13 18:00:00  29.9 -92.4      60.0         0
119696  2019192N29274  BARRY  2019-07-13 21:00:00  30.2 -92.6      55.0         0
119697  2019192N29274  BARRY  2019-07-14 00:00:00  30.4 -92.8      50.0         0
119698  2019192N29274  BARRY  2019-07-14 03:00:00  30.7 -93.0      45.0         0
119699  2019192N29274  BARRY  2019-07-14 06:00:00  31.0 -93.2      40.0         0
119700  201

In [7]:
import pandas as pd
import numpy as np
import os

BASE_DIR = ""
RAW_PATH = f"{BASE_DIR}/data/raw/noaa_ibtracs_raw.csv"
INTERIM_PATH = f"{BASE_DIR}/data/interim/"
os.makedirs(INTERIM_PATH, exist_ok=True)

# ── Country bounding boxes ─────────────────────────────────────────────────
# Format: ISO3 -> (lat_min, lat_max, lon_min, lon_max)
# Boxes are intentionally generous — storms within ~150km of coast
# are assigned to that country. Overlap is intentional for island nations.
COUNTRY_BOXES = {
    "BLZ": (15.8, 18.5, -89.2, -87.5),   # Belize
    "COL": (1.0,  13.5, -79.0, -66.9),   # Colombia (Caribbean coast)
    "CRI": (8.0,  11.2, -86.0, -82.5),   # Costa Rica
    "CUB": (19.6, 23.5, -85.0, -74.0),   # Cuba
    "DOM": (17.4, 20.1, -72.0, -68.3),   # Dominican Republic
    "SLV": (13.1, 14.5, -90.2, -87.6),   # El Salvador
    "GTM": (13.7, 17.8, -92.2, -88.2),   # Guatemala
    "GUY": (1.2,   8.6, -61.4, -56.5),   # Guyana
    "HTI": (17.9, 20.1, -74.5, -71.6),   # Haiti
    "HND": (12.9, 16.5, -89.4, -83.1),   # Honduras
    "JAM": (17.6, 18.6, -78.4, -76.2),   # Jamaica
    "NIC": (10.7, 15.0, -87.7, -82.6),   # Nicaragua
    "PAN": (7.2,  9.7,  -83.1, -77.2),   # Panama
    "PRI": (17.8, 18.6, -67.3, -65.2),   # Puerto Rico
    "SUR": (1.8,   6.0, -58.1, -53.9),   # Suriname
    "TTO": (9.9,  11.4, -61.9, -60.5),   # Trinidad and Tobago
    "VEN": (0.7,  12.2, -73.4, -59.8),   # Venezuela
    "BHS": (20.9, 27.3, -79.3, -72.7),   # Bahamas
    "BRB": (13.0, 13.4, -59.7, -59.4),   # Barbados
}

COUNTRY_NAMES = {
    "BLZ": "Belize", "COL": "Colombia", "CRI": "Costa Rica",
    "CUB": "Cuba", "DOM": "Dominican Republic", "SLV": "El Salvador",
    "GTM": "Guatemala", "GUY": "Guyana", "HTI": "Haiti",
    "HND": "Honduras", "JAM": "Jamaica", "NIC": "Nicaragua",
    "PAN": "Panama", "PRI": "Puerto Rico", "SUR": "Suriname",
    "TTO": "Trinidad and Tobago", "VEN": "Venezuela",
    "BHS": "Bahamas", "BRB": "Barbados",
}

COUNTRY_ROLE = {
    "COL": "pressure_input", "VEN": "pressure_input",
}

# SSHS category labels
SSHS_LABELS = {
    -4: "disturbance", -3: "depression", -2: "subtropical_depression",
    -1: "subtropical_storm", 0: "tropical_storm",
    1: "cat1", 2: "cat2", 3: "cat3", 4: "cat4", 5: "cat5",
}


def load_raw(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=[1], low_memory=False, na_values=[" ", ""])
    print(f"  Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
    return df


def filter_years(df: pd.DataFrame) -> pd.DataFrame:
    """Keep 2019–2024 only."""
    df = df[df["SEASON"].astype(str).str[:4].astype(int).between(2019, 2024)].copy()
    print(f"  After year filter (2019–2024): {len(df)} rows, {df['SID'].nunique()} storms")
    return df


def filter_storm_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep relevant storm natures:
      TS = tropical system  ← primary
      DS = disturbance      ← weak but trackable
      SS = subtropical      ← keep, relevant for Caribbean
      ET = extratropical    ← drop, no longer tropical threat
    """
    keep_natures = ["TS", "DS", "SS"]
    df = df[df["NATURE"].isin(keep_natures)].copy()
    print(f"  After nature filter (TS/DS/SS): {len(df)} rows")
    return df


def select_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only columns needed for country assignment and intensity."""
    keep = [
        "SID", "SEASON", "NAME", "ISO_TIME",
        "NATURE", "LAT", "LON",
        "USA_WIND", "USA_SSHS",
        "DIST2LAND", "LANDFALL",
    ]
    df = df[keep].copy()

    # Parse datetime
    df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"])
    df["year"] = df["ISO_TIME"].dt.year
    df["month"] = df["ISO_TIME"].dt.month

    # Clean types
    df["LAT"] = pd.to_numeric(df["LAT"], errors="coerce")
    df["LON"] = pd.to_numeric(df["LON"], errors="coerce")
    df["USA_WIND"] = pd.to_numeric(df["USA_WIND"], errors="coerce")
    df["USA_SSHS"] = pd.to_numeric(df["USA_SSHS"], errors="coerce")
    df["DIST2LAND"] = pd.to_numeric(df["DIST2LAND"], errors="coerce")
    df["LANDFALL"] = pd.to_numeric(df["LANDFALL"], errors="coerce")

    return df


def assign_countries(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign each storm observation to countries using bounding boxes.
    One observation can match multiple countries (island proximity).
    Returns long-format dataframe with one row per observation-country pair.
    """
    records = []

    for iso3, (lat_min, lat_max, lon_min, lon_max) in COUNTRY_BOXES.items():
        mask = (
            df["LAT"].between(lat_min, lat_max) &
            df["LON"].between(lon_min, lon_max)
        )
        matched = df[mask].copy()
        if len(matched) > 0:
            matched["ISO3"] = iso3
            matched["Country"] = COUNTRY_NAMES[iso3]
            matched["country_role"] = COUNTRY_ROLE.get(iso3, "tier1")
            records.append(matched)

    df_assigned = pd.concat(records, ignore_index=True)
    print(f"  Storm observations assigned to countries: {len(df_assigned)}")
    print(f"  Countries with storm activity: {df_assigned['ISO3'].nunique()}")
    return df_assigned


def build_storm_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse track observations to one row per storm per country.
    Captures peak intensity and whether the storm made landfall.
    """
    summary = df.groupby(["SID", "ISO3", "Country", "country_role",
                          "year", "month", "NAME"]).agg(
        peak_wind_knots=("USA_WIND", "max"),
        peak_sshs=("USA_SSHS", "max"),
        min_dist2land=("DIST2LAND", "min"),
        made_landfall=("LANDFALL", lambda x: int((x == 0).any())),
        track_observations=("ISO_TIME", "count"),
    ).reset_index()

    # Add SSHS label
    summary["sshs_label"] = summary["peak_sshs"].map(SSHS_LABELS).fillna("unknown")

    # Storm intensity score (0–1) based on peak wind
    # Max observed wind in dataset is 160 knots
    summary["wind_intensity_score"] = (
        summary["peak_wind_knots"] / 160
    ).clip(0, 1).round(4)

    # Year-month key
    summary["year_month"] = (
        summary["year"].astype(str) + "-" +
        summary["month"].astype(str).str.zfill(2)
    )

    print(f"  Storm-country records: {len(summary)}")
    return summary


def aggregate_to_country_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate storm summaries to country-month level.

    For each country-month:
      - storm_count          : number of distinct storms
      - max_wind_knots       : peak wind across all storms
      - max_sshs             : highest Saffir-Simpson category
      - landfall_count       : storms that made landfall
      - max_wind_score       : 0-1 intensity score
      - had_major_hurricane  : 1 if any Cat 3+ storm
    """
    agg = df.groupby(["ISO3", "Country", "country_role",
                      "year_month", "year", "month"]).agg(
        storm_count=("SID", "nunique"),
        max_wind_knots=("peak_wind_knots", "max"),
        max_sshs=("peak_sshs", "max"),
        landfall_count=("made_landfall", "sum"),
        max_wind_score=("wind_intensity_score", "max"),
        storm_names=("NAME", lambda x: "|".join(sorted(x.dropna().unique()))),
    ).reset_index()

    # Major hurricane flag (Cat 3+)
    agg["had_major_hurricane"] = (agg["max_sshs"] >= 3).astype(int)

    # Sort
    agg = agg.sort_values(["ISO3", "year", "month"]).reset_index(drop=True)

    print(f"  Country-month records: {len(agg)}")
    print(f"  Major hurricane months: {agg['had_major_hurricane'].sum()}")
    return agg


def validate(df_summary: pd.DataFrame, df_cm: pd.DataFrame):
    print("\n── Validation ──────────────────────────────")

    # Year range
    assert df_summary["year"].between(2019, 2024).all()
    print("  ✅ Year range: 2019–2024")

    # Wind score range
    assert df_summary["wind_intensity_score"].between(0, 1).all()
    print("  ✅ Wind intensity scores: 0–1 range confirmed")

    # Check high-profile storms are present
    known_storms = ["ETA", "IOTA", "DORIAN", "IDA", "FIONA", "BERYL"]
    found = df_summary["NAME"].str.upper().isin(known_storms).sum()
    print(f"  ✅ Known major storms found: {found} observations")

    # Countries covered
    covered = set(df_cm["ISO3"].unique())
    all_countries = set(COUNTRY_BOXES.keys())
    missing = all_countries - covered
    if missing:
        print(f"  ⚠️  Countries with no storm activity: {missing}")
        print(f"     (Will appear as zeros in master grid — expected for inland/low-exposure)")
    else:
        print("  ✅ All countries have storm activity records")

    print("────────────────────────────────────────────\n")


def main():
    print("\n🔄 NOAA IBTrACS Cleaning Pipeline")
    print("=" * 45)

    print("\n[1/7] Loading raw data...")
    df = load_raw(RAW_PATH)

    print("\n[2/7] Filtering to 2019–2024...")
    df = filter_years(df)

    print("\n[3/7] Filtering storm types...")
    df = filter_storm_types(df)

    print("\n[4/7] Selecting columns...")
    df = select_columns(df)

    print("\n[5/7] Assigning storms to countries...")
    df_assigned = assign_countries(df)

    print("\n[6/7] Building storm summaries...")
    df_summary = build_storm_summary(df_assigned)

    print("\n[7/7] Aggregating to country-month...")
    df_cm = aggregate_to_country_month(df_summary)

    validate(df_summary, df_cm)

    # Save
    summary_path = os.path.join(INTERIM_PATH, "noaa_clean.csv")
    cm_path = os.path.join(INTERIM_PATH, "noaa_country_month.csv")

    df_summary.to_csv(summary_path, index=False)
    df_cm.to_csv(cm_path, index=False)

    print("\n── Output Summary ──────────────────────────")
    print(f"  noaa_clean.csv         : {len(df_summary)} storm-country records")
    print(f"  noaa_country_month.csv : {len(df_cm)} country-month records")
    print(f"  Unique storms          : {df_summary['SID'].nunique()}")
    print(f"  Countries covered      : {df_cm['ISO3'].nunique()}")
    print(f"  Major hurricane months : {df_cm['had_major_hurricane'].sum()}")
    print("────────────────────────────────────────────\n")

    return df_summary, df_cm


if __name__ == "__main__":
    df_summary, df_cm = main()


🔄 NOAA IBTrACS Cleaning Pipeline

[1/7] Loading raw data...
  Loaded: 127569 rows × 174 columns

[2/7] Filtering to 2019–2024...
  After year filter (2019–2024): 7182 rows, 130 storms

[3/7] Filtering storm types...
  After nature filter (TS/DS/SS): 6160 rows

[4/7] Selecting columns...

[5/7] Assigning storms to countries...
  Storm observations assigned to countries: 578
  Countries with storm activity: 16

[6/7] Building storm summaries...
  Storm-country records: 85

[7/7] Aggregating to country-month...
  Country-month records: 77
  Major hurricane months: 8

── Validation ──────────────────────────────
  ✅ Year range: 2019–2024
  ✅ Wind intensity scores: 0–1 range confirmed
  ✅ Known major storms found: 18 observations
  ⚠️  Countries with no storm activity: {'SUR', 'GUY', 'PAN'}
     (Will appear as zeros in master grid — expected for inland/low-exposure)
────────────────────────────────────────────


── Output Summary ──────────────────────────
  noaa_clean.csv         : 85 st